# Early-Stop Splitters on Iris Dataset

Test all early-stopping split strategies (secretary, prophet, MAB) for axis-aligned decision tree regression using the classical Iris dataset.

We use a **regression** setup: predict a continuous target (e.g. petal length) from the other features, so that the gain is MSE and all splitter methods are comparable.

In [ ]:
import time

import numpy as np
from sklearn.datasets import load_iris
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from treeple.tree import EarlyStopDecisionTreeRegressor

## Load Iris and set up regression task

Predict **petal length** from the other three features (sepal length, sepal width, petal width). Alternatively we could use all 4 features and predict a numeric encoding of the species.

In [ ]:
iris = load_iris()
X_full = np.asarray(iris.data, dtype=np.float32)
feature_names = iris.feature_names

# Regression: predict petal length (index 2) from the other three features
target_idx = 2  # petal length
feature_idx = [i for i in range(X_full.shape[1]) if i != target_idx]
X = X_full[:, feature_idx]
y = X_full[:, target_idx]

print("Features (X):", [feature_names[i] for i in feature_idx])
print("Target (y):", feature_names[target_idx])
print("X.shape:", X.shape)
print("y.shape:", y.shape)

In [ ]:
RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE
)
print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

## Define all splitter methods to test

- **best**: baseline (exhaustive search).
- **secretary** (S): 1/e exploration, then first split better than max.
- **secretary_par** (S+par): secretary on covariates, parametric threshold.
- **secretary_all** (S+all): secretary on covariates, reward = max over thresholds.
- **double_secretary** (S²): double secretary (covariates then thresholds).
- **prophet** (PI): prophet inequality from samples.
- **prophet_par** (PI+par): prophet with parametric distribution.
- **mab_all** (MAB+all): MAB for covariate, then all thresholds.
- **mab_secretary** (MAB+S): MAB for covariate, then secretary on thresholds.
- **mab_par** (MAB+par): MAB for covariate, then parametric threshold.

In [ ]:
SPLITTERS_TO_TEST = [
    "best",
    "secretary",
    "secretary_par",
    "secretary_all",
    "double_secretary",
    "prophet",
    "prophet_par",
    "mab_all",
    "mab_secretary",
    "mab_par",
]

## Train and evaluate each method

In [ ]:
results = []

for splitter in SPLITTERS_TO_TEST:
    # Parametric methods accept alpha via split_search
    split_search = {"alpha": 0.5} if splitter in ("secretary_par", "prophet_par", "mab_par") else None
    est = EarlyStopDecisionTreeRegressor(
        splitter=splitter,
        random_state=RANDOM_STATE,
        split_search=split_search,
    )
    t0 = time.perf_counter()
    est.fit(X_train, y_train)
    fit_time = time.perf_counter() - t0
    y_pred = est.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append({
        "splitter": splitter,
        "mse": mse,
        "r2": r2,
        "fit_time_sec": fit_time,
        "n_leaves": est.get_n_leaves(),
    })
    print(f"{splitter:20s}  MSE={mse:.6f}  R²={r2:.4f}  time={fit_time:.4f}s  leaves={est.get_n_leaves()}")

## Summary table

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df = df.set_index("splitter")
display(df.round(6))

## Optional: multiple runs for variance

Run each method with several random seeds to see mean and std of MSE/R² (especially for stochastic methods like secretary).

In [ ]:
n_repeats = 10
agg_results = []

for splitter in SPLITTERS_TO_TEST:
    split_search = {"alpha": 0.5} if splitter in ("secretary_par", "prophet_par", "mab_par") else None
    mse_list, r2_list, time_list = [], [], []
    for seed in range(RANDOM_STATE, RANDOM_STATE + n_repeats):
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed)
        est = EarlyStopDecisionTreeRegressor(
            splitter=splitter,
            random_state=seed,
            split_search=split_search,
        )
        t0 = time.perf_counter()
        est.fit(X_tr, y_tr)
        time_list.append(time.perf_counter() - t0)
        mse_list.append(mean_squared_error(y_te, est.predict(X_te)))
        r2_list.append(r2_score(y_te, est.predict(X_te)))
    agg_results.append({
        "splitter": splitter,
        "mse_mean": np.mean(mse_list),
        "mse_std": np.std(mse_list),
        "r2_mean": np.mean(r2_list),
        "r2_std": np.std(r2_list),
        "time_mean_sec": np.mean(time_list),
    })

agg_df = pd.DataFrame(agg_results).set_index("splitter")
display(agg_df.round(6))